In [ ]:
# Standard library
import json
import os
from pathlib import Path

# Visualization
import matplotlib.pyplot as plt
from matplotlib import ticker

# Core ML & audio stack
import librosa
import lightning.pytorch as pl
import numpy as np
import pandas as pd
import soundfile as sf
import torch

# HuggingFace datasets
from datasets import load_dataset

# ASR models & normalization
from nemo.collections.asr.models import ASRModel
from transformers.models.whisper.english_normalizer import EnglishTextNormalizer

# Training & experiment utilities
from loguru import logger
from nemo.utils import logging
from nemo.utils.exp_manager import exp_manager
from nemo.utils.trainer_utils import resolve_trainer_cfg
from omegaconf import OmegaConf, open_dict

# Project utilities
from asr_benchmark.config import PROJECT_ROOT
from asr_benchmark.nemo_adapter import (
    add_global_adapter_cfg,
    patch_transcribe_lhotse,
    update_model_cfg,
    update_model_config_to_support_adapter,
)
from asr_benchmark.score import english_spelling_normalizer, score_wer

In [ ]:
torch.set_float32_matmul_precision("high")
# Set SAMPLE to use a smaller subset of the data for faster iteration during development. Set it to None to use the full dataset.
SAMPLE = None

In [ ]:
DATASET_ID = "quinnlue/asr"

ds = load_dataset(DATASET_ID)
train_hf = ds["train"]
val_hf = ds["validation"]
test_hf = ds["test"]

logger.info(f"Loaded HuggingFace dataset '{DATASET_ID}' — "
            f"train: {len(train_hf)}, val: {len(val_hf)}, test: {len(test_hf)}")

In [ ]:
MANIFEST_DIR = PROJECT_ROOT / "data" / "processed" / "ortho_dataset"
AUDIO_CACHE = MANIFEST_DIR / "audio"
train_manifest_path = MANIFEST_DIR / "train_manifest.jsonl"
val_manifest_path = MANIFEST_DIR / "val_manifest.jsonl"
clip_max_duration_sec = 25.0

MANIFEST_DIR.mkdir(parents=True, exist_ok=True)
AUDIO_CACHE.mkdir(parents=True, exist_ok=True)

In [ ]:
def hf_split_to_manifest(split_dataset, manifest_path, max_duration=25.0, sample_n=None):
    """Convert a HuggingFace dataset split to a NeMo-format manifest JSONL.

    Audio arrays are written to .wav files in AUDIO_CACHE so that NeMo can
    read them by file path.  Clips longer than *max_duration* are skipped.
    """
    if sample_n:
        logger.info(f"Sampling {sample_n} utterances")
        split_dataset = split_dataset.select(range(min(sample_n, len(split_dataset))))

    records = []
    skipped = 0
    for idx, row in enumerate(split_dataset):
        dur = row["audio_duration_sec"]
        if dur > max_duration:
            skipped += 1
            continue
        uid = row.get("utterance_id", f"utt_{idx}")
        wav_path = AUDIO_CACHE / f"{uid}.wav"
        if not wav_path.exists():
            sf.write(
                str(wav_path),
                row["audio_path"]["array"],
                row["audio_path"]["sampling_rate"],
            )
        records.append({
            "audio_filepath": str(wav_path),
            "duration": dur,
            "text": row["orthographic_text"],
        })
    logger.info(
        f"Wrote {len(records)} entries to {manifest_path.name} "
        f"(skipped {skipped} clips > {max_duration}s)"
    )
    pd.DataFrame(records).to_json(manifest_path, orient="records", lines=True)
    return len(records)


n_train = hf_split_to_manifest(
    train_hf, train_manifest_path,
    max_duration=clip_max_duration_sec, sample_n=SAMPLE,
)
n_val = hf_split_to_manifest(
    val_hf, val_manifest_path,
    max_duration=clip_max_duration_sec, sample_n=SAMPLE,
)

In [ ]:
# ── Hardware-dependent settings ──────────────────────────────────────────────
# Adjust these to match your GPU memory and CPU cores.
DEVICES = 1
PRECISION = "bf16-mixed"
BATCH_SIZE = 32
NUM_WORKERS = 8

# ── Load NeMo adapter defaults ───────────────────────────────────────────────
yaml_path = PROJECT_ROOT / "asr_benchmark" / "assets" / "asr_adaptation.yaml"
cfg = OmegaConf.load(yaml_path)

# ── Training overrides ───────────────────────────────────────────────────────
overrides = OmegaConf.create(
    {
        "model": {
            "pretrained_model": "nvidia/parakeet-tdt-0.6b-v3",
            "adapter": {
                "adapter_name": "asr_children_orthographic",
                "adapter_module_name": "encoder",
                "linear": {"in_features": 1024},
            },
            "train_ds": {
                "manifest_filepath": str(train_manifest_path),
                "batch_size": BATCH_SIZE,
                "num_workers": NUM_WORKERS,
                "use_lhotse": False,
                "channel_selector": "average",
            },
            "validation_ds": {
                "manifest_filepath": str(val_manifest_path),
                "batch_size": BATCH_SIZE,
                "num_workers": NUM_WORKERS,
                "use_lhotse": False,
                "channel_selector": "average",
            },
            "optim": {
                "lr": 0.001,
                "weight_decay": 0.0,
            },
        },
        "trainer": {
            "devices": DEVICES,
            "precision": PRECISION,
            "strategy": "auto",
            "max_epochs": 1 if SAMPLE else None,
            "max_steps": -1 if SAMPLE else 5000,
            "val_check_interval": 1.0 if SAMPLE else 500,
            "enable_progress_bar": False,
        },
        "exp_manager": {
            "exp_dir": str(PROJECT_ROOT / "models" / "orthographic_benchmark_nemo"),
        },
    }
)

cfg = OmegaConf.merge(cfg, overrides)

## 3. Define Trainer

The Trainer orchestrates the training loop across devices, delegating tensor operations to PyTorch's backend. We initiate the trainer with the `OmegaConf` config object we made above, and then set up an experiment manager to handle logging, checkpoint saving, and saving artifacts to disk.

Note we use the cell magic [`%%capture`](https://ipython.readthedocs.io/en/9.2.0/interactive/magics.html#cellmagic-capture) below to hide long cell outputs for readability.

In [ ]:
%%capture
trainer = pl.Trainer(**resolve_trainer_cfg(cfg.trainer))
exp_log_dir = exp_manager(trainer, cfg.get("exp_manager", None))

## 4. Data and Model Setup

We load the pretrained model by fetching the model from our config first and then patching it for adapter support. Then, we load the model weights.

In [ ]:
%%capture
model_cfg = ASRModel.from_pretrained(cfg.model.pretrained_model, return_config=True)
update_model_config_to_support_adapter(model_cfg, cfg)
model = ASRModel.from_pretrained(
    cfg.model.pretrained_model,
    override_config_path=model_cfg,
    trainer=trainer,
)

We disable the CUDA graph decoder because it is incompatible with current PyTorch (2.10).

In [ ]:
%%capture
with open_dict(model.cfg):
    model.cfg.decoding.greedy.use_cuda_graph_decoder = False
model.change_decoding_strategy(model.cfg.decoding)

Next, we prepare our data by merging our parameter overrides (batch size, num workers, etc.) into the model's built-in data config. Note that while keys not present in the original config are dropped, whitelisted keys are always injected.

In [ ]:
cfg.model.train_ds = update_model_cfg(model.cfg.train_ds, cfg.model.train_ds)
model.setup_training_data(cfg.model.train_ds)

cfg.model.validation_ds = update_model_cfg(model.cfg.validation_ds, cfg.model.validation_ds)
model.setup_multiple_validation_data(cfg.model.validation_ds)

We also need to set the optimization function, which controls how the model’s weights are updated using gradients to minimize the loss and improve performance during training. We use AdamW with cosine annealing schedule and 10% warmup ratio as a starting point.

In [ ]:
%%capture
model.setup_optimization(cfg.model.optim)

In [ ]:
# Configure spec augmentation from config if available, otherwise disable it.
if "spec_augment" in cfg.model:
    model.spec_augmentation = model.from_config_dict(cfg.model.spec_augment)
else:
    model.spec_augmentation = None
    del model.cfg.spec_augment

## 5. Set up Adapter

An adapter is a small neural network module that learns to transform the pretrained model's internal representations specifically for child speech. Rather than retraining the entire model, we insert lightweight adapters into each layer of the encoder, the part of the model that processes audio features.

We add a linear adapter to every encoder layer, then freeze the base model and unfreeze only the adapter weights. This keeps training efficient—only ~0.1% of parameters are trainable, while (hopefully) allowing the adapter to learn the acoustic and linguistic patterns unique to children's voices.

In [ ]:
%%capture
with open_dict(cfg.model.adapter):
    adapter_name = cfg.model.adapter.pop("adapter_name")
    adapter_type = cfg.model.adapter.pop("adapter_type")
    adapter_module_name = cfg.model.adapter.pop("adapter_module_name", None)
    adapter_state_dict_name = cfg.model.adapter.pop("adapter_state_dict_name", None)

    adapter_type_cfg = cfg.model.adapter[adapter_type]

    if adapter_module_name is not None and ":" not in adapter_name:
        adapter_name = f"{adapter_module_name}:{adapter_name}"

    adapter_global_cfg = cfg.model.adapter.pop(model.adapter_global_cfg_key, None)
    if adapter_global_cfg is not None:
        add_global_adapter_cfg(model, adapter_global_cfg)

model.add_adapter(adapter_name, cfg=adapter_type_cfg)
assert model.is_adapter_available()

model.set_enabled_adapters(enabled=False)
model.set_enabled_adapters(adapter_name, enabled=True)

model.freeze()
model = model.train()
model.unfreeze_enabled_adapters()

In [ ]:
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,} ({100 * trainable_params / total_params:.2f}%)")

## 6. Train

Let's adapt the pretrained model to children's voices!

In [ ]:
%%capture
trainer.fit(model)

We've hidden the training cell outputs to avoid printing many lines of logs. Below are a few key snippets of the log output that we saw when training:
```bash
  | Name              | Type                              | Params | Mode 
--------------------------------------------------------------------------------
0 | preprocessor      | AudioToMelSpectrogramPreprocessor | 0      | train
1 | encoder           | ConformerEncoderAdapter           | 610 M  | train
2 | decoder           | RNNTDecoder                       | 7.2 M  | train
3 | joint             | RNNTJoint                         | 1.7 M  | train
4 | loss              | RNNTLoss                          | 0      | train
5 | spec_augmentation | SpectrogramAugmentation           | 0      | train
6 | wer               | WER                               | 0      | train
--------------------------------------------------------------------------------
1.6 M     Trainable params
617 M     Non-trainable params
619 M     Total params
2,477.792 Total estimated model params size (MB)

Epoch 0, global step 500: 'val_wer' reached 0.20008 (best 0.20008), saving model to '/models/orthographic_benchmark_nemo/ASR-Adapter/2026-02-26_04-58-33/checkpoints/ASR-Adapter--val_wer=0.2001-epoch=0.ckpt' as top 5
...
Epoch 0, global step 5000: 'val_wer' reached 0.16359 (best 0.16359), saving model to '/models/orthographic_benchmark_nemo/ASR-Adapter/2026-02-26_04-58-33/checkpoints/ASR-Adapter--val_wer=0.1636-epoch=0.ckpt' as top 5
`Trainer.fit` stopped: `max_steps=5000` reached.
```

Because we configured `max_steps` to be 5,000, we stopped training before we completed the first epoch.

We need to save the final results. We save the adapter weights to a standalone file alongside the NeMo checkpoints.

In [ ]:
if adapter_state_dict_name is not None:
    state_path = exp_log_dir if exp_log_dir is not None else os.getcwd()
    ckpt_path = os.path.join(state_path, "checkpoints")
    if os.path.exists(ckpt_path):
        state_path = ckpt_path
    state_path = os.path.join(state_path, adapter_state_dict_name)

    model.save_adapters(str(state_path))

## 7. Evaluation
Now it's time to assess how well our adapted model performs. We load the best trained checkpoint from disk and run inference across the entire validation set. The model transcribes each audio clip, and we compare these predictions against the reference transcriptions using Word Error Rate (WER). The WER calculation is copied exactly from the [runtime repository](https://github.com/drivendataorg/childrens-speech-recognition-runtime/blob/main/metric/score.py).

In [ ]:
%%capture
nemo_ckpts = sorted((exp_log_dir / "checkpoints").glob("*.nemo"))
if not nemo_ckpts:
    raise FileNotFoundError(f"No .nemo checkpoints found in {exp_log_dir}/checkpoints/")

best_ckpt = nemo_ckpts[-1]
print(f"Loading checkpoint: {best_ckpt}")
eval_model = ASRModel.restore_from(best_ckpt, map_location="cuda")

with open_dict(eval_model.cfg):
    eval_model.cfg.decoding.greedy.use_cuda_graph_decoder = False
eval_model.change_decoding_strategy(eval_model.cfg.decoding)

patch_transcribe_lhotse(eval_model)

In [ ]:
val_manifest_path = cfg.model.validation_ds.manifest_filepath
with open(val_manifest_path) as f:
    val_entries = [json.loads(line) for line in f]

audio_files = [e["audio_filepath"] for e in val_entries]
references = [e["text"] for e in val_entries]

print(f"Running inference on {len(audio_files)} validation utterances...")
raw = eval_model.transcribe(
    audio_files, batch_size=BATCH_SIZE, channel_selector="average", verbose=False
)
if isinstance(raw, tuple):
    raw = raw[0]

predictions = [h.text if hasattr(h, "text") else h for h in raw]

Before scoring, let's remove any example where the normalized label is an empty string.

In [ ]:
normalizer = EnglishTextNormalizer(english_spelling_normalizer)
filtered = [(r, p) for r, p in zip(references, predictions) if normalizer(r) != ""]

references, predictions = zip(*filtered)

wer = score_wer(references, predictions)

print(f"Validation WER: {wer:.4f}")

print("\nSample predictions:")
for ref, pred in zip(references[:5], predictions[:5]):
    print(f"  REF:  {ref}")
    print(f"  PRED: {pred}")
    print()

Our Validation WER after adapting the model is 0.15! 

___
# Step 3: Make your submission

Since this is a code execution competition, we will submit our model weights and code rather than predictions. See the [code submission format](https://www.drivendata.org/competitions/308/childrens-word-asr/page/978/) webpage for more information.

The general steps to follow:

1. Develop inference code
2. Test your submission locally
3. Package submission
4. Make a smoke test submission
5. Once you have successfully debugged your submission, submit it for scoring on the full test set!


## Develop Inference Code

We need to set up a repository with a `main.py` Python script which performs inference in the [competition execution environment](https://github.com/drivendataorg/childrens-speech-recognition-runtime/tree/main) and writes our predictions to the required output file.  During code execution, our submission will be unzipped and run in the cloud compute cluster. The container will run your `main.py` script.

Our code must write a JSON Lines (JSONL) file containing one prediction per utterance.

Each line must include:
- `utterance_id`
- `orthographic_text`: UTF-8, standard English transcription of the utterance
The submission should be written to ./submission/submission.jsonl relative to the working directory.

See more details in the [code submission format](https://www.drivendata.org/competitions/308/childrens-word-asr/page/978/) webpage and in the [example submission](https://github.com/drivendataorg/childrens-speech-recognition-runtime/tree/main/examples/word/parakeet). 

In our `main.py`, we load the trained adapter checkpoint from disk, restore the NeMo model, and run inference on all test utterances in batches. The script reads audio file paths from the test manifest, transcribes them using the adapted model, and writes the predicted transcriptions to the submission file in the required format. We sort utterances by duration before inference (longest first) to improve GPU memory efficiency during batching.

See `orthographic_submission/main.py` for the details.

## Test Submission Locally

You should first and foremost test your submission locally. This is a great way to work out any bugs and ensure that your model performs inference successfully. See the [runtime repository's README](https://github.com/drivendataorg/childrens-speech-recognition-runtime/tree/main?tab=readme-ov-file#testing-a-submission-locally) for further instructions.


This repository provides a useful justfile command to run the trained model on a few sample files.

```bash
# Run inference using data-demo/word/ to test orthographic submission
test-orthographic:
    uv run orthographic_submission/main.py models/orthographic_benchmark_nemo/ASR-Adapter-best.nemo data-demo/word/utterance_metadata.jsonl
```

## Package Submission

Now we will package up our model and inference code into a zip file for predicting on the test set in the runtime container. This repository provides a justfile command to do this. The justfile finds the latest models weights, and then creates a zipfile combining those model weights with `/orthographic_submissions/main.py`.

```bash
pack-orthographic:
    #!/usr/bin/env bash
    set -euo pipefail
    latest=$(ls -td models/orthographic_benchmark_nemo/ASR-Adapter/*/checkpoints/ASR-Adapter.nemo | head -1)
    ln -sf "${latest#models/orthographic_benchmark_nemo/}" models/orthographic_benchmark_nemo/ASR-Adapter-best.nemo
    echo "Updated ASR-Adapter-best.nemo -> $latest"
    rm -f orthographic_submission.zip
    (cd orthographic_submission && zip -r ../orthographic_submission.zip main.py)
    (cd models/orthographic_benchmark_nemo && zip -r ../../orthographic_submission.zip ASR-Adapter-best.nemo)
```

Another packing command is available on the runtime repo's orthographic [example](https://github.com/drivendataorg/childrens-speech-recognition-runtime/blob/main/examples/word/parakeet/pack_submission.sh).

## Make a Smoke Test Submission

We provide a "smoke test" environment that replicates the test inference runtime but runs only on a small set of audio files. In the smoke test runtime, data/ contains 9,000 audio files from the training set. 

Let's submit our submission.zip to a smoke test on the platform.

![Smoke Test Submission](images/smoke_test_ortho.png)

After hitting "Submit" you can see the job in the queue—it will progress from "Uploading" to "Pending" to "Starting" to "Running":

![Smoke Test Code Jobs](images/smoke_test_jobs_ortho.png)

Once your submission reaches "Completed", head on over to the "Submissions" tab to see your smoke test score.

## Submit!

After you've made sure a smoke test submission runs without error, you're ready to submit the real deal! This adaptation of `parakeet-tdt-0.6b-v2` results in a .2370 WER on the full test set. 

![Full Submission](images/submission_ortho.png)

We encourage you to also be mindful of the submission limit (3 per 7 days at most) and others' code jobs. Canceled jobs do not count against the submission limit.

This is of course just one of the approaches you could take for this challenge. Some resources that may be helpful in getting started are:
 * The Hugging Face [Automatic Speech Recognition](https://huggingface.co/learn/audio-course/en/chapter5/introduction) unit of their Audio Course.
 * The Hugging Face [Open ASR Leaderboard](https://huggingface.co/spaces/hf-audio/open_asr_leaderboard). 
 * A [Parakeet fine-tuning implementation](https://github.com/Deep-unlearning/Finetune-Parakeet) from Deep-unlearning.
 * NeMo example [ASR adaptation](https://github.com/NVIDIA-NeMo/NeMo/tree/main/examples/asr/asr_adapters) (which has a similar implementation as this blog post).
 
If you want to share any of your findings or have questions, feel free to post on the [community forum](https://community.drivendata.org/c/childrens-asr/109). 

Good luck!